# Pareto front over volume fraction (Sec. 4.2)

Generates `figures/pareto_vf_structures.pdf` from `results/2_pareto_vf/`.

The compliance values are from `history.csv`, computed during optimization on the continuous filtered field.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from plot_utils import FIGURES, RESULTS, draw_cantilever_bcs, load_design, plot_field, specimen_cmap

RUN_DIR = RESULTS / "2_pareto_vf"
VFS = [0.3, 0.4, 0.5, 0.6, 0.7, 0.8]

compliance = [
    np.genfromtxt(RUN_DIR / f"data_creep_{int(vf * 100)}" / "history.csv", delimiter=",", names=True)["J"][-1]
    for vf in VFS
]

In [ ]:
FIG_W, FIG_H = 6.0, 3.6
fig, ax = plt.subplots(figsize=(FIG_W, FIG_H))
ax.plot(VFS, compliance, "o-", color="C0", lw=1.2, ms=5, zorder=3)
ax.set_xlabel("Volume fraction")
ax.set_ylabel("Creep compliance [J]")
ax.set_title("Pareto frontier of optimized structures")
ax.grid(True, alpha=0.3)
ax.margins(x=0.08)
y_lo, y_hi = ax.get_ylim()
ax.set_ylim(y_lo, y_lo + 1.12 * (y_hi - y_lo))  # Headroom for the insets

# Size the insets for the structures' 2:1 aspect ratio.
thumb_w = 0.18
thumb_h = thumb_w * (FIG_W / FIG_H) * (100 / 200)

# Inset centres, in axes fractions, staggered in the open region above the front.
centers = {
    0.3: (0.26, 0.80),
    0.4: (0.37, 0.55),
    0.5: (0.50, 0.80),
    0.6: (0.62, 0.55),
    0.7: (0.76, 0.80),
    0.8: (0.90, 0.55),
}

for vf, J in zip(VFS, compliance):
    cx, cy = centers[vf]
    x0, y0 = cx - thumb_w / 2, cy - thumb_h / 2

    vtu_path = sorted((RUN_DIR / f"data_creep_{int(vf * 100)}" / "vtk").glob("sol_*.vtu"))[-1]
    design = load_design(vtu_path, vf)
    inset = ax.inset_axes([x0, y0, thumb_w, thumb_h])
    plot_field(inset, design, design.binary, cmap=specimen_cmap, vmin=0.0, vmax=1.0, border=False)
    draw_cantilever_bcs(inset, design.points, inside=True, arrow_below=0.1, arrowhead=15)
    inset.set_title(f"vf = {vf}", fontsize=7, pad=2)

    # Connect each inset to its point: from the left edge for vf = 0.3, else the bottom edge.
    anchor = (x0, cy) if vf == 0.3 else (cx, y0)
    ax.annotate("", xy=(vf, J), xytext=anchor, textcoords=ax.transAxes,
                arrowprops=dict(arrowstyle="-", color="0.55", lw=0.7, linestyle="--"), zorder=2)

fig.savefig(FIGURES / "pareto_vf_structures.pdf", bbox_inches="tight")
plt.show()